In [1]:
from eris.core.registry import list_encoders, get_encoder, get_encoder_info
from eris.core.types import StructureInput

# CLI

In [2]:
!python -m eris --help

                                                                               
 Usage: python -m eris [OPTIONS] COMMAND [ARGS]...                             
                                                                               
┌─ Options ───────────────────────────────────────────────────────────────────┐
│ --help          Show this message and exit.                                 │
└─────────────────────────────────────────────────────────────────────────────┘
┌─ Commands ──────────────────────────────────────────────────────────────────┐
│ cache     Cache utilities.                                                  │
│ encode    Encode single structures into Eris artifacts.                     │
│ batch     Batch encoding utilities.                                         │
│ info      Artifact inspection utilities.                                    │
│ encoders  List registered encoders.                                         │
└───────────────────────────────────────

In [3]:
!python -m eris encoders list

                                 Eris Encoders                                 
┌──────────────────────────┬────────┬─────────┬───────────────────────────────┐
│ Name                     │ Type   │ Level   │ Description                   │
├──────────────────────────┼────────┼─────────┼───────────────────────────────┤
│ atom_knn_graph           │ graph  │ atom    │ Atom kNN graph with baseline  │
│                          │        │         │ deterministic node/edge/graph │
│                          │        │         │ features                      │
│                          │        │         │ (elem/role/bfactor/occ,       │
│                          │        │         │ dist+RBF, seq separation,     │
│                          │        │         │ local flags, graph stats).    │
│ atom_point_cloud         │ cloud  │ atom    │ Atom point cloud.             │
│ contact_map_bins_residue │ matrix │ ca      │ Binned CA contact map (NxN    │
│                          │        │   

In [4]:
!python -m eris encode run \
--input "../tests/data/1AQ2.pdb" \
--encoder atom_knn_graph \
--out "../out" 

[05/19/26 15:17:44] INFO     eris.core.base_encoder.base_en base_encoder.py:197
                             coder | INFO | Loading                            
                             structure                                         
                    INFO     eris.core.base_encoder.base_enc base_encoder.py:49
                             oder | INFO | Structure loaded.                   
..\out\1AQ2\atom_knn_graph


# Encoders

In [5]:
list_encoders()

['atom_knn_graph',
 'atom_point_cloud',
 'contact_map_bins_residue',
 'contact_map_residue',
 'distogram_bins_residue',
 'distogram_residue',
 'residue_contact_graph',
 'residue_knn_graph',
 'residue_point_cloud',
 'voxel_grid_ca']

In [6]:
DATA = "../tests/data/1AQ2.pdb"

# Graph

## Atom KNN graph

In [7]:
print(get_encoder_info("atom_knn_graph"))


EncoderInfo(name='atom_knn_graph', type='graph', level='atom', description='Atom kNN graph with baseline deterministic node/edge/graph features (elem/role/bfactor/occ, dist+RBF, seq separation, local flags, graph stats).', params_schema={'k': {'type': 'int', 'default': 16}, 'include_self_loops': {'type': 'bool', 'default': False}, 'include_hetero': {'type': 'bool', 'default': False}, 'include_h': {'type': 'bool', 'default': False}, 'altloc': {'type': 'str', 'default': 'first'}, 'amino_acids_only': {'type': 'bool', 'default': False}, 'rbf_k': {'type': 'int', 'default': 16}, 'rbf_dmax': {'type': 'float', 'default': 20.0}, 'local_cutoff': {'type': 'int', 'default': 4}})


In [8]:
enc = get_encoder("atom_knn_graph", k=5)
art = enc.encode(StructureInput(source=str(DATA)))
enc

AtomKNNGraph(use_cache=True, cache=None, center=True, scale=None, k=5, cutoff=12.0, symmetric=True, include_self_loops=False, node_attr_mode='elem+role+bfactor+occ', rbf_k=16, rbf_dmax=20.0, local_cutoff=4)

## Residue KNN graph

In [9]:
print(get_encoder_info("residue_knn_graph"))

EncoderInfo(name='residue_knn_graph', type='graph', level='residue', description='Residue kNN graph (CA) with baseline node/edge/graph features (aa_index, physchem, plddt/bfactor, RBF, seq separation, local flags).', params_schema={'k': {'type': 'int', 'default': 16}, 'symmetrize': {'type': 'bool', 'default': False}, 'include_self_loops': {'type': 'bool', 'default': False}})


In [10]:
enc = get_encoder("residue_knn_graph", k=10, knn_mode="global")
art = enc.encode(StructureInput(source=str(DATA)))
enc

ResidueKNNGraph(name='residue_knn_graph', use_cache=True, cache=None, center=True, scale=None, knn_mode='global', k=10, cutoff=None, symmetric=True, include_self_loops=False)

## Residue contact graph

In [11]:
print(get_encoder_info("residue_contact_graph"))

EncoderInfo(name='residue_contact_graph', type='graph', level='residue', description='Residue contact graph (CA) with baseline node/edge/graph features (aa_index, physchem, plddt/bfactor, RBF, seq separation, local flags).', params_schema={'cutoff': {'type': 'float', 'default': 8.0}, 'include_self_loops': {'type': 'bool', 'default': False}})


In [12]:
enc = get_encoder("residue_contact_graph", cutoff=10)
art = enc.encode(StructureInput(source=str(DATA)))
enc

ResidueContactGraph(representation=<eris.core.representations.graph_rep.GraphRepresentation object at 0x0000014FCE35B920>, use_cache=True, cache=None, center=True, scale=None, include_self_loops=False, cutoff=10)

# Point Cloud

## Atom PC

In [13]:
print(get_encoder_info("atom_point_cloud"))

EncoderInfo(name='atom_point_cloud', type='cloud', level='atom', description='Atom point cloud.', params_schema={'include_hetero': {'type': 'bool', 'default': False}, 'include_h': {'type': 'bool', 'default': False}, 'altloc': {'type': 'str', 'default': 'first'}, 'amino_acids_only': {'type': 'bool', 'default': False}, 'mode': {'type': 'str', 'default': 'none'}})


In [14]:
enc = get_encoder("atom_point_cloud")
art = enc.encode(StructureInput(source=str(DATA)))
enc

AtomPointCloud(use_cache=True, cache=None, center=True, scale=None, node_attr_mode='elem+role+bfactor+occ')

## Residue PC

In [15]:
print(get_encoder_info("residue_point_cloud"))

EncoderInfo(name='residue_point_cloud', type='cloud', level='residue', description='Residue point cloud from CA coordinates (+ residue ids, aa_index, plddt/bfactor if available).', params_schema={'center': {'type': 'bool', 'default': True}, 'scale': {'type': ['null', 'float'], 'default': None}})


In [16]:
enc = get_encoder("residue_point_cloud")
art = enc.encode(StructureInput(source=str(DATA)))
enc

ResiduePointCloudCA(use_cache=True, cache=None, center=True, scale=None, node_attr_mode='onehot+physchem+plddt+bfactor')

# Voxel/Grid

In [17]:
print(get_encoder_info("voxel_grid_ca"))

EncoderInfo(name='voxel_grid_ca', type='grid', level='ca', description='Voxel grid representation around CA coordinates (occupancy, density, pLDDT, B-factor, aa_onehot channels).', params_schema={'grid_size': {'type': 'int', 'default': 32}, 'voxel_size': {'type': ['null', 'float'], 'default': None}, 'fit': {'type': 'bool', 'default': True}, 'margin': {'type': 'float', 'default': 2.0}, 'center': {'type': 'bool', 'default': True}, 'channels': {'type': 'str', 'default': 'occupancy', 'allowed': ['occupancy', 'density', 'plddt', 'bfactor', 'aa_onehot']}, 'sigma': {'type': 'float', 'default': 1.0}, 'pool': {'type': 'str', 'default': 'sum', 'allowed': ['sum', 'mean', 'max']}, 'clamp': {'type': 'bool', 'default': False}, 'dtype': {'type': 'str', 'default': 'float32', 'allowed': ['float32', 'float16']}})


In [18]:
enc = get_encoder("voxel_grid_ca")
art = enc.encode(StructureInput(source=str(DATA)))
enc

VoxelGridCA(name='voxel_grid_ca', use_cache=True, cache=None, center=True, scale=None, grid_size=64, dtype='float32', channels='occupancy', fit=True, margin=4.0, voxel_size=None, sigma=1.0, pool='max', clamp=True)

# Geometry

## Contact Map Bins

In [19]:
print(get_encoder_info("contact_map_bins_residue"))

EncoderInfo(name='contact_map_bins_residue', type='matrix', level='ca', description='Binned CA contact map (NxN classes; contact/no-contact + distance bins).', params_schema={'include_self': {'type': 'bool', 'default': False}, 'add_no_contact_class': {'type': 'bool', 'default': True}, 'dmin': {'type': 'float', 'default': 2.0}, 'dmax': {'type': 'float', 'default': 20.0}, 'num_bins': {'type': 'int', 'default': 32}, 'dtype': {'type': 'str', 'default': 'uint8'}})


In [20]:
enc = get_encoder("contact_map_bins_residue")
art = enc.encode(StructureInput(source=str(DATA)))
enc

ContactMapBinsCA(use_cache=True, cache=None, center=True, scale=None, edges=None, dmin=2.0, dmax=20.0, num_bins=36, include_self=False, add_no_contact_class=True, dtype='uint8')

## Contact map

In [21]:
print(get_encoder_info("contact_map_residue"))

EncoderInfo(name='contact_map_residue', type='matrix', level='ca', description='Binary CA contact map (NxN) under a distance cutoff.', params_schema={'cutoff': {'type': 'float', 'default': 8.0}, 'include_self': {'type': 'bool', 'default': False}, 'dtype': {'type': 'str', 'default': 'bool'}})


In [22]:
enc = get_encoder("contact_map_residue")
art = enc.encode(StructureInput(source=str(DATA)))
enc

ContactMapCA(use_cache=True, cache=None, center=True, scale=None, cutoff=8.0, include_self=False, dtype='uint8')

## Distogram Bins

In [23]:
print(get_encoder_info("distogram_bins_residue"))

EncoderInfo(name='distogram_bins_residue', type='matrix', level='ca', description='Binned CA distogram (int NxN classes with configurable bin edges).', params_schema={'include_self': {'type': 'bool', 'default': True}, 'add_no_contact_class': {'type': 'bool', 'default': False}, 'dmin': {'type': 'float', 'default': 2.0}, 'dmax': {'type': 'float', 'default': 20.0}, 'num_bins': {'type': 'int', 'default': 32}, 'dtype': {'type': 'str', 'default': 'uint8'}})


In [24]:
enc = get_encoder("distogram_bins_residue")
art = enc.encode(StructureInput(source=str(DATA)))
enc

DistogramBinsCA(use_cache=True, cache=None, center=True, scale=None, edges=None, dmin=2.0, dmax=20.0, num_bins=36, include_self=True, add_no_contact_class=True, dtype='uint8')

## Distogram

In [25]:
print(get_encoder_info("distogram_residue"))

EncoderInfo(name='distogram_residue', type='matrix', level='ca', description='CA distogram (float NxN pairwise distances).', params_schema={'include_self': {'type': 'bool', 'default': True}})


In [26]:
enc = get_encoder("distogram_residue")
art = enc.encode(StructureInput(source=str(DATA)))
enc

DistogramCA(use_cache=True, cache=None, center=True, scale=None, include_self=True)